# 🏆 Reconstruct v6 FINAL - Precision Feature Engineering

**The Definitive Pipeline:**
- REAL data from raw parquets
- Precision trends per patient ID
- 50/50 balanced training
- 19 columns output

**Output:**
- `master_train_v6_engineered.parquet`
- `master_test_v6_engineered.parquet`

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import numpy as np
import os

print(f"Polars: {pl.__version__}")
print("🔧 Using streaming mode for 130M+ rows")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
RAW_PATH = f"{BASE_PATH}/raw"
OUTPUT_PATH = f"{BASE_PATH}/processed"

FILES = {
    "train_raw": f"{RAW_PATH}/metabonet_public_train.parquet",
    "test_raw": f"{RAW_PATH}/metabonet_public_test.parquet",
    "train_out": f"{OUTPUT_PATH}/master_train_v6_engineered.parquet",
    "test_out": f"{OUTPUT_PATH}/master_test_v6_engineered.parquet"
}

for name, path in FILES.items():
    if "raw" in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Data Loading (Lazy Mode)

In [ ]:
REQUIRED_COLS = [
    'id', 'date', 'CGM', 'bolus', 'carbs', 'basal',
    'age', 'weight', 'gender', 'insulin_type_bolus'
]

def scan_raw(path, label):
    """Scan raw parquet with lazy execution."""
    print(f"\n📂 Scanning {label}...")
    lf = pl.scan_parquet(path)
    
    # Get available columns
    available = [c for c in REQUIRED_COLS if c in lf.columns]
    missing = [c for c in REQUIRED_COLS if c not in lf.columns]
    
    print(f"   ✅ Available: {len(available)} cols")
    if missing:
        print(f"   ⚠️ Missing: {missing}")
    
    # Select and pre-clean
    lf = lf.select(available).with_columns([
        pl.col('carbs').fill_null(0.0),
        pl.col('basal').fill_null(0.0)
    ])
    
    return lf, available

lf_train, train_cols = scan_raw(FILES['train_raw'], 'TRAIN')
lf_test, test_cols = scan_raw(FILES['test_raw'], 'TEST')

## 3. Advanced Feature Engineering (The Engine)

In [ ]:
def add_temporal_features(lf):
    """
    A. Temporal Features from REAL date.
    """
    if 'date' not in lf.columns:
        print("   ⚠️ 'date' missing - skipping temporal")
        return lf
    
    return lf.with_columns([
        # 24h cycle sin/cos
        (2 * np.pi * pl.col('date').dt.hour() / 24).sin().alias('time_sin'),
        (2 * np.pi * pl.col('date').dt.hour() / 24).cos().alias('time_cos'),
        # Day of week (0=Mon, 6=Sun)
        pl.col('date').dt.weekday().alias('day_of_week'),
        # Weekend flag
        (pl.col('date').dt.weekday() >= 5).cast(pl.Int8).alias('is_weekend')
    ])

print("✅ add_temporal_features defined")

In [ ]:
def add_trend_features(lf):
    """
    B. High-Precision Trends per Patient ID.
    """
    if 'id' not in lf.columns:
        print("   ⚠️ 'id' missing - using global diff")
        return lf.with_columns([
            pl.col('CGM').diff().fill_null(0).alias('cgm_delta_5min'),
            pl.col('CGM').diff(n=3).fill_null(0).alias('cgm_delta_15min')
        ])
    
    # Sort by patient and time for accurate diffs
    return lf.sort(['id', 'date']).with_columns([
        # 5-min delta (1 row = 5 mins typically)
        pl.col('CGM').diff(n=1).over('id').fill_null(0).alias('cgm_delta_5min'),
        # 15-min delta (3 rows)
        pl.col('CGM').diff(n=3).over('id').fill_null(0).alias('cgm_delta_15min')
    ])

print("✅ add_trend_features defined")

In [ ]:
def add_categorical_encodings(lf):
    """
    C. Categorical Encodings from REAL values.
    """
    # Gender: Male=0, Female=1
    if 'gender' in lf.columns:
        lf = lf.with_columns(
            pl.when(pl.col('gender') == 'Male').then(0)
            .when(pl.col('gender') == 'Female').then(1)
            .otherwise(0)
            .alias('gender_encoded')
        )
    else:
        lf = lf.with_columns(pl.lit(0).alias('gender_encoded'))
    
    # Insulin type: Map to integers (0-7)
    if 'insulin_type_bolus' in lf.columns:
        lf = lf.with_columns(
            pl.when(pl.col('insulin_type_bolus') == 'Humalog').then(0)
            .when(pl.col('insulin_type_bolus') == 'NovoRapid').then(1)
            .when(pl.col('insulin_type_bolus') == 'Apidra').then(2)
            .when(pl.col('insulin_type_bolus') == 'Fiasp').then(3)
            .when(pl.col('insulin_type_bolus') == 'Lyumjev').then(4)
            .when(pl.col('insulin_type_bolus') == 'Actrapid').then(5)
            .when(pl.col('insulin_type_bolus') == 'Humulin').then(6)
            .otherwise(7)
            .alias('insulin_type_encoded')
        )
    else:
        lf = lf.with_columns(pl.lit(0).alias('insulin_type_encoded'))
    
    return lf

print("✅ add_categorical_encodings defined")

In [ ]:
def apply_feature_engineering(lf):
    """Apply full feature engineering pipeline."""
    lf = add_temporal_features(lf)
    lf = add_trend_features(lf)
    lf = add_categorical_encodings(lf)
    return lf

print("✅ apply_feature_engineering defined")

## 4. Toxic Filtering (Clinical Safety)

In [ ]:
def get_toxic_mask():
    """Clinically dangerous rows to DROP."""
    return (
        (pl.col('CGM') < 110) & 
        (pl.col('carbs') == 0) & 
        (pl.col('bolus') > 0)
    )

def apply_toxic_filter(lf):
    """Remove toxic rows."""
    return lf.filter(~get_toxic_mask())

print("✅ Toxic filter defined")

## 5. Process TRAIN (Feature Engineering + Balance)

In [ ]:
print("🔧 Processing TRAIN...")

# Apply feature engineering and toxic filter
lf_train_processed = apply_feature_engineering(lf_train)
lf_train_processed = apply_toxic_filter(lf_train_processed)

# Collect with streaming
print("   Collecting (streaming)...")
df_train = lf_train_processed.collect(streaming=True)
print(f"   Collected: {len(df_train):,} rows")

In [ ]:
# Balance 50/50
print("\n⚖️ Balancing TRAIN...")

pool_action = df_train.filter(pl.col('bolus') > 0)
pool_safe = df_train.filter(pl.col('bolus') == 0)

n_action = len(pool_action)
n_safe = len(pool_safe)

print(f"   Action (bolus>0): {n_action:,}")
print(f"   Safe (bolus=0): {n_safe:,}")

In [ ]:
# Balance Safe pool by CGM range (50% low, 50% high)
target_safe = min(n_action, n_safe)

pool_safe_low = pool_safe.filter(pl.col('CGM') < 120)
pool_safe_high = pool_safe.filter(pl.col('CGM') >= 120)

sample_low = min(target_safe // 2, len(pool_safe_low))
sample_high = min(target_safe - sample_low, len(pool_safe_high))

print(f"   Sampling Safe: {sample_low:,} low + {sample_high:,} high")

if sample_low > 0 and sample_low < len(pool_safe_low):
    pool_safe_low = pool_safe_low.sample(n=sample_low, seed=42)
if sample_high > 0 and sample_high < len(pool_safe_high):
    pool_safe_high = pool_safe_high.sample(n=sample_high, seed=42)

pool_safe_balanced = pl.concat([pool_safe_low, pool_safe_high])

In [ ]:
# Merge and shuffle
df_train_final = pl.concat([pool_action, pool_safe_balanced])
df_train_final = df_train_final.sample(fraction=1.0, seed=42, shuffle=True)

# Cast to Float32
numeric_cols = [c for c in df_train_final.columns 
                if df_train_final[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
                and c not in ['id']]
df_train_final = df_train_final.cast({c: pl.Float32 for c in numeric_cols})

print(f"\n✅ TRAIN final: {len(df_train_final):,} rows, {len(df_train_final.columns)} cols")

## 6. Process TEST (Feature Engineering only, NO Balance)

In [ ]:
print("🔧 Processing TEST...")

# Apply feature engineering and toxic filter
lf_test_processed = apply_feature_engineering(lf_test)
lf_test_processed = apply_toxic_filter(lf_test_processed)

# Collect with streaming
print("   Collecting (streaming)...")
df_test_final = lf_test_processed.collect(streaming=True)

# Cast to Float32
numeric_cols = [c for c in df_test_final.columns 
                if df_test_final[c].dtype in [pl.Float64, pl.Int64, pl.Int32]
                and c not in ['id']]
df_test_final = df_test_final.cast({c: pl.Float32 for c in numeric_cols})

print(f"\n✅ TEST final: {len(df_test_final):,} rows, {len(df_test_final.columns)} cols")

## 7. Export

In [ ]:
print(f"\n💾 Saving TRAIN: {FILES['train_out']}")
df_train_final.write_parquet(FILES['train_out'])
print("✅ Saved!")

print(f"\n💾 Saving TEST: {FILES['test_out']}")
df_test_final.write_parquet(FILES['test_out'])
print("✅ Saved!")

## 8. Verification

In [ ]:
MODEL_FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]

print("\n🔍 MODEL FEATURES CHECK:")
print("=" * 50)
for col in MODEL_FEATURES:
    train_ok = "✅" if col in df_train_final.columns else "❌"
    test_ok = "✅" if col in df_test_final.columns else "❌"
    print(f"   {col:25} Train: {train_ok} | Test: {test_ok}")

In [ ]:
print("\n🔍 REAL DATA VALIDATION:")
print("=" * 50)

# Check time_sin range
if 'time_sin' in df_train_final.columns:
    ts_min = df_train_final['time_sin'].min()
    ts_max = df_train_final['time_sin'].max()
    print(f"   time_sin: [{ts_min:.3f}, {ts_max:.3f}] (expect: [-1, 1])")

# Check cgm_delta variance
if 'cgm_delta_5min' in df_train_final.columns:
    delta_std = df_train_final['cgm_delta_5min'].std()
    print(f"   cgm_delta_5min std: {delta_std:.3f} (>0 = real data)")

# Check patient count
if 'id' in df_train_final.columns:
    n_patients = df_train_final['id'].n_unique()
    print(f"   Unique patients: {n_patients:,}")
    print("   ✅ Real patient IDs preserved!")

In [ ]:
print("\n" + "=" * 70)
print("📋 FINAL SUMMARY")
print("=" * 70)

train_action = len(df_train_final.filter(pl.col('bolus') > 0))
train_safe = len(df_train_final.filter(pl.col('bolus') == 0))

test_action = len(df_test_final.filter(pl.col('bolus') > 0))
test_safe = len(df_test_final.filter(pl.col('bolus') == 0))

print(f"""
TRAIN (Balanced 50/50):
   Total: {len(df_train_final):,} rows
   Action: {train_action:,} ({train_action/len(df_train_final)*100:.1f}%)
   Safe: {train_safe:,} ({train_safe/len(df_train_final)*100:.1f}%)
   Columns: {len(df_train_final.columns)}

TEST (Real-World Distribution):
   Total: {len(df_test_final):,} rows
   Action: {test_action:,} ({test_action/len(df_test_final)*100:.1f}%)
   Safe: {test_safe:,} ({test_safe/len(df_test_final)*100:.1f}%)
   Columns: {len(df_test_final.columns)}

OUTPUT FILES:
   ✅ {FILES['train_out']}
   ✅ {FILES['test_out']}

COLUMNS:
   {df_train_final.columns}
""")

print("=" * 70)
print("🏆 v6 FINAL ENGINEERED - 100% REAL DATA")
print("=" * 70)